# 🏁 Semantic Search and RAG: A Comprehensive Tutorial

## Introduction to Modern Information Retrieval

Welcome to this comprehensive tutorial on **Semantic Search** and **Retrieval-Augmented Generation (RAG)**! 

In this notebook, we'll explore how modern AI systems understand and retrieve information beyond simple keyword matching. We'll build a complete information retrieval system using Formula 1 data as our playground.

### What You'll Learn:

1. **🔍 Semantic Search**: How embeddings capture meaning beyond keywords
2. **📊 Vector Databases**: Efficient similarity search with FAISS
3. **🔤 Keyword Search**: Traditional BM25 algorithm and its strengths
4. **🔄 Hybrid Search**: Combining semantic and keyword approaches
5. **🤖 RAG Systems**: Using retrieved context to generate better answers
6. **⚡ Local vs Cloud**: Comparing different model deployment strategies

### Prerequisites:
- Basic understanding of Python and machine learning
- Familiarity with vectors and embeddings (we'll review these concepts)
- Interest in information retrieval and NLP

Let's dive in! 🚀


## 📦 Setting Up Our Environment

First, let's import all the libraries we'll need for this tutorial. Each serves a specific purpose in our information retrieval pipeline:

- **`cohere`**: For generating high-quality embeddings and reranking
- **`faiss`**: Facebook's efficient similarity search library
- **`numpy`** & **`pandas`**: Data manipulation and numerical operations
- **`nltk`**: Natural language processing tools for text preprocessing
- **`rank_bm25`**: Implementation of the BM25 ranking algorithm
- **`langchain`**: Framework for building LLM applications and RAG systems


In [127]:
# Core libraries for semantic search and embeddings
import cohere  # Cloud-based embedding and reranking service
import numpy as np  # Numerical operations for vector computations
import pandas as pd  # Data manipulation and analysis
from tqdm import tqdm  # Progress bars for long-running operations

# Natural language processing
import nltk  # Text processing and tokenization
nltk.download('punkt')  # Download sentence tokenization data

# Additional imports we'll use later
import faiss  # Efficient similarity search and clustering
from rank_bm25 import BM25Okapi  # BM25 keyword search algorithm
from sklearn.feature_extraction import _stop_words  # Stop word removal
import string  # String manipulation utilities

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


[nltk_data] Downloading package punkt to /Users/mayia/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## 📚 Our Dataset: Formula 1 Knowledge Base

For this tutorial, we'll use a rich text about Formula 1 history as our knowledge base. This text contains various types of information:

- **Historical facts** (dates, championships, records)
- **Technical details** (car specifications, regulations)
- **Financial information** (team costs, ownership)
- **Biographical data** (drivers, achievements)

This diversity makes it perfect for testing different search strategies, as some queries will benefit from semantic understanding while others require precise keyword matching.

### Why Formula 1 Data?

F1 data is ideal for demonstrating search capabilities because:
- It contains **factual information** that can be verified
- It has **temporal relationships** (years, seasons, eras)
- It includes **technical terminology** that might not appear in training data
- It has **numerical data** (costs, statistics) that requires precise retrieval

In [173]:
# Our comprehensive Formula 1 knowledge base
# This text contains diverse information types that will help us demonstrate
# different search strategies and their effectiveness

formula_one_knowledge_base = """Formula One (F1) is the highest class of worldwide racing for open-wheel single-seater formula racing cars sanctioned by the Fédération Internationale de l'Automobile (FIA). The FIA Formula One World Championship has been one of the world's premier forms of motorsport since its inaugural running in 1950 and is often considered to be the pinnacle of motorsport. The word formula in the name refers to the set of rules all participant cars must follow. A Formula One season consists of a series of races, known as Grands Prix. Grands Prix take place in multiple countries and continents on either purpose-built circuits or closed roads.
A points scoring system is used at Grands Prix to determine two annual World Championships: one for the drivers, and one for the constructors—now synonymous with teams. Each driver must hold a valid Super Licence, the highest class of racing licence the FIA issues, and the races must be held on Grade One tracks, the highest grade rating the FIA issues for tracks.
Formula One cars are the world's fastest regulated road-course racing cars, owing to high cornering speeds achieved by generating large amounts of aerodynamic downforce, most of which is generated by front and rear wings, as well as underbody tunnels. The cars depend on electronics, aerodynamics, suspension, and tyres. Traction control, launch control, automatic shifting, and other electronic driving aids were first banned in 1994. They were briefly reintroduced in 2001, and have more recently been banned since 2004 and 2008, respectively.
With the average annual cost of running a team— designing, building, and maintaining cars; staff payroll; transport—at approximately £220 million, Formula One's financial and political battles are widely reported. The Formula One Group is owned by Liberty Media, which acquired it in 2017 from a private-equity firm.
Formula One originated from the World Manufacturers' Championship and European Drivers' Championship. The formula is a set of rules that all participants' cars must follow. Formula One was a formula agreed upon in 1946 to officially become effective in 1947. The first Grand Prix in accordance with the new regulations was the 1946 Turin Grand Prix, anticipating the formula's official start. Before World War II, a number of Grand Prix racing organisations made suggestions for a new championship to replace the European Championship, but due to the suspension of racing during the conflict, the new International Formula for cars did not become formalised until 1946, to become effective in 1947. The new World Championship was instituted to commence in 1950.
The first world championship race, the 1950 British Grand Prix, took place at Silverstone Circuit in the United Kingdom on 13 May 1950. Giuseppe Farina, competing for Alfa Romeo, won the first Drivers' World Championship, narrowly defeating his teammate Juan Manuel Fangio. Fangio won the championship in 1951, 1954, 1955, 1956, and 1957. This set the record for the most World Championships won by a single driver, a record that stood for 46 years until Michael Schumacher won his sixth championship in 2003.
A Constructors' Championship was added in the 1958 season. Stirling Moss, despite often being regarded as one of the greatest Formula One drivers in the 1950s and 1960s, never won the Formula One championship. Between 1955 and 1961, Moss finished second in the championship four times and third the other three times. Fangio won 24 of the 52 races he entered—still the record for the highest Formula One winning percentage by an individual driver. National championships existed in South Africa and the UK in the 1960s and 1970s. Promoters held non-championship Formula One events for many years. Due to the increasing cost of competition, the last of these was held in 1983.
This era featured teams managed by road-car manufacturers, such as Alfa Romeo, Ferrari, Mercedes-Benz and Maserati. The first seasons featured prewar cars like Alfa Romeo's 158, which were front-engined, with narrow tyres and 1.5-litre supercharged or 4.5-litre naturally aspirated engines. The 1952 and 1953 seasons were run to Formula Two regulations, for smaller, less powerful cars, due to concerns over the dearth of Formula One cars. When a new Formula One formula for engines limited to 2.5 litres was reinstated for the 1954 world championship, Mercedes-Benz introduced its W196, which featured things never seen on Formula One cars before, such as desmodromic valves, fuel injection, and enclosed streamlined bodywork. Mercedes drivers won the championship for the next two years, before the team withdrew from all motorsport competitions due to the 1955 Le Mans disaster."""


### 🔧 Text Preprocessing: Breaking Down Our Knowledge Base

Before we can search through our text, we need to **chunk** it into smaller, manageable pieces. This is crucial for several reasons:

#### Why Do We Chunk Text?

1. **🎯 Granular Retrieval**: Smaller chunks allow us to retrieve more specific, relevant information
2. **⚡ Efficient Processing**: Embedding models work better with shorter text segments
3. **📐 Context Windows**: LLMs have limited context windows; smaller chunks fit better
4. **🔍 Precision**: We can find the exact paragraph that answers a question

#### Chunking Strategies:

- **📄 Sentence-based**: Split by sentences (what we'll use)
- **📏 Fixed-size**: Split by character/word count
- **🏗️ Semantic**: Split by topics or sections
- **📖 Document-structure**: Split by paragraphs, headers, etc.

For this tutorial, we'll use **sentence-based chunking** with NLTK's sentence tokenizer.


In [130]:

# Use NLTK's sentence tokenizer to split our knowledge base into individual sentences
# This creates our "document chunks" - each sentence becomes a searchable unit
document_chunks = nltk.sent_tokenize(formula_one_knowledge_base)

# Let's analyze our chunking results
print(f"📋 Chunking Results:")
print(f"   • Number of chunks created: {len(document_chunks)}")
print(f"   • Average chunk length: {np.mean([len(chunk) for chunk in document_chunks]):.1f} characters")
print(f"   • Shortest chunk: {min([len(chunk) for chunk in document_chunks])} characters")
print(f"   • Longest chunk: {max([len(chunk) for chunk in document_chunks])} characters")

print(f"\n📝 Sample chunks:")
for i, chunk in enumerate(document_chunks[:3]):  # Show first 3 chunks
    print(f"   Chunk {i+1}: '{chunk[:80]}...'")
    
print(f"\n💡 Pro tip: Good chunk size balances specificity with context!")
print(f"   • Too small: Loses context and meaning")
print(f"   • Too large: Less precise retrieval, harder to process")

📋 Chunking Results:
   • Number of chunks created: 35
   • Average chunk length: 133.2 characters
   • Shortest chunk: 58 characters
   • Longest chunk: 305 characters

📝 Sample chunks:
   Chunk 1: 'Formula One (F1) is the highest class of worldwide racing for open-wheel single-...'
   Chunk 2: 'The FIA Formula One World Championship has been one of the world's premier forms...'
   Chunk 3: 'The word formula in the name refers to the set of rules all participant cars mus...'

💡 Pro tip: Good chunk size balances specificity with context!
   • Too small: Loses context and meaning
   • Too large: Less precise retrieval, harder to process


## 🧠 Semantic Search: Understanding Meaning Beyond Keywords

**Semantic search** revolutionizes how we find information by understanding the **meaning** behind queries rather than just matching keywords.

### 🔍 Traditional vs Semantic Search

| **Traditional (Keyword) Search** | **Semantic Search** |
|----------------------------------|-------------------|
| Matches exact words/phrases      | Understands concepts and meaning |
| `"fast car"` → finds `"fast car"` | `"fast car"` → finds `"high-speed vehicle"` |
| Boolean operators (AND, OR, NOT) | Natural language queries |
| Requires precise terminology     | Works with synonyms and paraphrases |

### 🎯 How Semantic Search Works

1. **🔢 Text Embeddings**: Convert text into high-dimensional vectors
2. **📐 Vector Space**: Similar meanings cluster together in vector space
3. **📏 Similarity Calculation**: Use distance metrics (cosine, L2) to find similar vectors
4. **🔄 Retrieval**: Return chunks with highest similarity scores

### 🏗️ Our Semantic Search Pipeline

```
📝 Text Chunk → 🤖 Embedding Model → 📊 Vector → 💾 Vector Database
                                                      ↓
🔍 Query → 🤖 Same Model → 📊 Query Vector → 📏 Similarity Search → 📋 Results
```

### 🌟 Advantages of Semantic Search

- **🎯 Intent Understanding**: Finds answers even with different wording
- **🌍 Language Flexibility**: Works across languages and dialects  
- **🧠 Conceptual Matching**: Connects related concepts
- **📚 Domain Adaptation**: Learns domain-specific terminology

Let's build our semantic search system!

### 🔑 Setting Up Cohere for Embeddings

**Cohere** provides state-of-the-art embedding models that convert text into meaningful vector representations. 

#### Why Cohere for Embeddings?

- **🎯 High Quality**: Trained on diverse, high-quality data
- **⚡ Speed**: Fast API responses for real-time applications
- **🔧 Specialized**: Different input types for documents vs queries
- **🌍 Multilingual**: Supports 100+ languages

#### Input Types in Cohere:

- **`search_document`**: For encoding documents/chunks in your knowledge base
- **`search_query`**: For encoding user queries
- **`classification`**: For text classification tasks
- **`clustering`**: For document clustering applications

Using different input types helps the model optimize embeddings for their specific use case!


In [131]:
# 🔐 Cohere API Configuration
# In production, NEVER hardcode API keys! Use environment variables instead.
# For this tutorial, we'll show both approaches:

# Method 1: Environment variable (RECOMMENDED)
import os
# cohere_api_key = os.getenv('COHERE_API_KEY')  # Uncomment in production

# Method 2: Direct assignment (ONLY for tutorials/testing)
cohere_api_key = ""  # Replace with your key

# Initialize the Cohere client
cohere_client = cohere.Client(cohere_api_key)

print("✅ Cohere client initialized successfully!")
print(f"🔧 Using API key ending in: ...{cohere_api_key[-8:]}")
print(f"📡 Ready to generate embeddings!")

✅ Cohere client initialized successfully!
🔧 Using API key ending in: ...cplxtKc0
📡 Ready to generate embeddings!


### 🚀 Generating Document Embeddings

Now we'll convert each text chunk into a high-dimensional vector representation. This is the core of semantic search!

#### What happens during embedding generation?

1. **📝 Input**: Each sentence from our F1 knowledge base
2. **🤖 Processing**: Cohere's neural network processes the text
3. **📊 Output**: A 4096-dimensional vector for each sentence
4. **💾 Storage**: Vectors stored as numpy arrays for efficient computation

#### Technical Details:

- **Model**: Cohere's latest embedding model (embed-english-v3.0)
- **Dimensions**: 4096 (higher dimensions = more nuanced representations)
- **Input Type**: `search_document` (optimized for document encoding)
- **Batch Processing**: All chunks processed together for efficiency


In [132]:
# 🔄 Generate embeddings for all document chunks
# This is where the magic happens - text becomes numbers!

print("🚀 Generating embeddings for document chunks...")
print(f"📊 Processing {len(document_chunks)} text chunks...")

# Call Cohere's embedding API
embedding_response = cohere_client.embed(
    texts=document_chunks,  # Our list of F1 text chunks
    input_type='search_document',  # Optimized for document encoding
    model='embed-english-v3.0'  # Latest Cohere embedding model
)

# Convert to numpy array for efficient mathematical operations
document_embeddings = np.array(embedding_response.embeddings)

# 📊 Analyze our embedding matrix
print(f"\n✅ Embeddings generated successfully!")
print(f"📐 Embedding matrix shape: {document_embeddings.shape}")
print(f"   • {document_embeddings.shape[0]} documents")
print(f"   • {document_embeddings.shape[1]} dimensions per embedding")
print(f"💾 Memory usage: {document_embeddings.nbytes / 1024 / 1024:.2f} MB")

# 🔍 Quick peek at embedding properties
print(f"\n🔬 Embedding Analysis:")
print(f"   • Value range: [{document_embeddings.min():.3f}, {document_embeddings.max():.3f}]")
print(f"   • Mean magnitude: {np.linalg.norm(document_embeddings, axis=1).mean():.3f}")
print(f"   • Standard deviation: {document_embeddings.std():.3f}")

🚀 Generating embeddings for document chunks...
📊 Processing 35 text chunks...

✅ Embeddings generated successfully!
📐 Embedding matrix shape: (35, 1024)
   • 35 documents
   • 1024 dimensions per embedding
💾 Memory usage: 0.27 MB

🔬 Embedding Analysis:
   • Value range: [-0.170, 0.141]
   • Mean magnitude: 1.000
   • Standard deviation: 0.031


### 🗄️ Building Our Vector Database with FAISS

**FAISS** (Facebook AI Similarity Search) is a library for efficient similarity search and clustering of dense vectors. It's the backbone of many production vector databases!

#### Why FAISS?

- **⚡ Speed**: Highly optimized for similarity search
- **📈 Scalability**: Handles millions of vectors efficiently  
- **🔧 Flexibility**: Multiple indexing algorithms
- **🎯 Accuracy**: Exact and approximate search options
- **💾 Memory Efficient**: Smart compression techniques

#### FAISS Index Types:

| **Index Type** | **Use Case** | **Speed** | **Accuracy** |
|----------------|--------------|-----------|--------------|
| `IndexFlatL2` | Exact search, small datasets | Medium | 100% |
| `IndexFlatIP` | Exact search, inner product | Medium | 100% |
| `IndexIVFFlat` | Large datasets, good balance | Fast | ~99% |
| `IndexHNSW` | Real-time apps, memory efficient | Very Fast | ~95% |

For our tutorial, we'll use `IndexFlatL2` (exact L2 distance search) since we have a small dataset and want perfect accuracy.


In [133]:
# 🏗️ Building our FAISS vector database
# This creates a searchable index of our document embeddings

print("🔨 Building FAISS vector database...")

# Get the dimensionality of our embeddings (should be 4096 for Cohere)
embedding_dimensions = document_embeddings.shape[1]
print(f"📐 Vector dimensions: {embedding_dimensions}")

# Create a FAISS index for exact L2 (Euclidean) distance search
# L2 distance measures how "far apart" vectors are in high-dimensional space
vector_index = faiss.IndexFlatL2(embedding_dimensions)

print(f"🏗️ Created {type(vector_index).__name__} index")
print(f"   • Distance metric: L2 (Euclidean)")
print(f"   • Search type: Exact (100% accuracy)")

# Add all our document embeddings to the index
# FAISS requires float32 format for memory efficiency
vector_index.add(np.float32(document_embeddings))

print(f"\n✅ Vector database built successfully!")
print(f"📊 Database statistics:")
print(f"   • Total vectors indexed: {vector_index.ntotal}")
print(f"   • Index size: {vector_index.ntotal * embedding_dimensions * 4 / 1024 / 1024:.2f} MB")
print(f"   • Ready for similarity search! 🔍")

🔨 Building FAISS vector database...
📐 Vector dimensions: 1024
🏗️ Created IndexFlatL2 index
   • Distance metric: L2 (Euclidean)
   • Search type: Exact (100% accuracy)

✅ Vector database built successfully!
📊 Database statistics:
   • Total vectors indexed: 35
   • Index size: 0.14 MB
   • Ready for similarity search! 🔍


### 🔎 Building Our Semantic Search Function

Now comes the exciting part - creating a function that can understand the meaning behind questions and find relevant answers!

#### How Our Search Function Works:

1. **🎯 Query Encoding**: Convert user question to vector using same model
2. **📏 Similarity Calculation**: Find closest document vectors using L2 distance
3. **🏆 Ranking**: Sort results by similarity score (lower distance = higher relevance)
4. **📋 Results**: Return top-k most relevant text chunks

#### Distance Metrics Explained:

- **L2 Distance**: Euclidean distance in high-dimensional space
  - Lower values = more similar
  - Range: [0, ∞)
  - Good for: General semantic similarity

- **Cosine Similarity**: Angle between vectors (alternative approach)
  - Higher values = more similar  
  - Range: [-1, 1]
  - Good for: Document similarity regardless of length


In [154]:
def semantic_search_engine(user_query, top_k_results=3, verbose=True):
    """
    🔍 Semantic Search Engine
    
    Finds the most semantically similar document chunks to a user query.
    
    Parameters:
    -----------
    user_query : str
        The question or search query from the user
    top_k_results : int, default=3
        Number of most similar results to return
    verbose : bool, default=True
        Whether to print detailed search information
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with columns: ['chunk_text', 'similarity_score', 'chunk_id']
        Sorted by similarity (lower scores = more similar)
    """
    
    print(f"🔍 Semantic Search Query: '{user_query}'")
    if verbose:
        print(f"📊 Searching through {len(document_chunks)} document chunks...")
    
    # Step 1: Convert user query to embedding vector
    # Use 'search_query' input type for optimal query encoding
    query_embedding_response = cohere_client.embed(
        texts=[user_query],
        input_type="search_query",  # Optimized for query encoding
        model='embed-english-v3.0'
    )
    query_vector = np.array(query_embedding_response.embeddings[0])
    
    if verbose:
        print(f"🎯 Query embedding generated: {query_vector.shape[0]} dimensions")
    
    # Step 2: Search for similar vectors in our FAISS index
    # Returns: distances (similarity scores) and indices of most similar documents
    similarity_distances, most_similar_indices = vector_index.search(
        np.float32([query_vector]),  # Query vector (must be 2D array)
        top_k_results  # Number of results to return
    )
    
    # Step 3: Extract the relevant information
    # Get the actual text chunks and their similarity scores
    document_chunks_array = np.array(document_chunks)
    relevant_chunks = document_chunks_array[most_similar_indices[0]]
    similarity_scores = similarity_distances[0]
    
    # Step 4: Create structured results
    search_results = pd.DataFrame({
        'chunk_text': relevant_chunks,
        'similarity_score': similarity_scores,
        'chunk_id': most_similar_indices[0]
    })
    
    if verbose:
        print(f"✅ Found {len(search_results)} relevant results")
        print(f"📏 Similarity scores range: {similarity_scores.min():.3f} - {similarity_scores.max():.3f}")
        print(f"💡 Lower scores = higher similarity")
        
        print(f"\n🏆 Top Results:")
        for idx, row in search_results.iterrows():
            print(f"   {idx+1}. Score: {row['similarity_score']:.3f}")
            print(f"      Text: \"{row['chunk_text'][:100]}...\"")
            print()
    
    else:
        print(f"\n🏆 Top Results:")
        for idx, row in search_results.iterrows():
            print(f"      Text: \"{row['chunk_text'][:100]}...\"")
            print()

    return search_results
    

## Testing Our Semantic Search System

Let's evaluate our semantic search with carefully designed test queries. Each query tests different aspects of semantic understanding:

**Historical Facts**: Queries about specific events, dates, and people  
**Technical Details**: Questions about car specifications and regulations  
**Financial Information**: Cost-related queries  
**Temporal Reasoning**: Understanding time periods and sequences


### Lets test:
“Who was the most successful F1 driver in the 1950s?”
→ Should return info about Juan Manuel Fangio.

“When was the Constructors' Championship introduced?”
→ Look for phrase “1958 season”.

“Who won the first world championship race?”
→ Should match “Giuseppe Farina”.

“Which manufacturer introduced new engine technology in the 1950s?”
→ Should bring up Mercedes-Benz and its W196 innovations.

“Why did Mercedes leave Formula One in the 1950s?”
→ Looks for the 1955 Le Mans disaster explanation.

“What are the safety requirements for tracks in Formula One?”
→ Should find reference to “Grade One tracks.”

“How are modern F1 cars able to go so fast around corners?”
→ Answer should mention aerodynamic downforce and wings.

“How much does running a team cost?”
→ Should hit on “£220 million”.

“What is the minimum licence a driver needs?”
→ Hits on “Super Licence”.

“What cars were used in the 1952 and 1953 seasons?”
→ Should find “Formula Two regulations”.

In [156]:
# Define our test queries
# These queries test various aspects of semantic understanding

evaluation_queries = [
    "Who was the most successful F1 driver in the 1950s?",
    "When was the Constructors' Championship introduced?", 
    "Who won the first world championship race?",
    "Which manufacturer introduced new engine technology in the 1950s?",
    "Why did Mercedes leave Formula One in the 1950s?",
    "What are the safety requirements for tracks in Formula One?",
    "How are modern F1 cars able to go so fast around corners?",
    "How much does running a team cost?",
    "What is the minimum licence a driver needs?",
    "What cars were used in the 1952 and 1953 seasons?"
]

In [158]:
# Test our semantic search system with all evaluation queries

for i, query in enumerate(evaluation_queries):
    results = semantic_search_engine(query, top_k_results=1, verbose=False)
    print("-" * 40)


🔍 Semantic Search Query: 'Who was the most successful F1 driver in the 1950s?'

🏆 Top Results:
      Text: "Stirling Moss, despite often being regarded as one of the greatest Formula One drivers in the 1950s ..."

----------------------------------------
🔍 Semantic Search Query: 'When was the Constructors' Championship introduced?'

🏆 Top Results:
      Text: "A Constructors' Championship was added in the 1958 season...."

----------------------------------------
🔍 Semantic Search Query: 'Who won the first world championship race?'

🏆 Top Results:
      Text: "The first world championship race, the 1950 British Grand Prix, took place at Silverstone Circuit in..."

----------------------------------------
🔍 Semantic Search Query: 'Which manufacturer introduced new engine technology in the 1950s?'

🏆 Top Results:
      Text: "When a new Formula One formula for engines limited to 2.5 litres was reinstated for the 1954 world c..."

----------------------------------------
🔍 Semantic Search

## Keyword Search with BM25

While semantic search excels at understanding meaning, keyword search remains valuable for:

**Exact Matches**: Finding specific terms, numbers, or names  
**High Precision**: When you know the exact terminology  
**Complementary Retrieval**: Different from semantic search results

### BM25 Algorithm

BM25 (Best Matching 25) is a probabilistic ranking function based on:
- **Term Frequency (TF)**: How often a term appears in a document
- **Inverse Document Frequency (IDF)**: How rare a term is across the corpus
- **Document Length Normalization**: Accounts for varying document sizes

**Formula**: BM25(q,d) = Σ IDF(qi) × f(qi,d) × (k1+1) / (f(qi,d) + k1 × (1-b+b × |d|/avgdl))

where k1 and b are tuning parameters.


### Text Preprocessing for BM25

Before building our BM25 index, we need to preprocess the text by:
1. **Tokenization**: Split text into individual words
2. **Normalization**: Convert to lowercase
3. **Stop word removal**: Remove common words (the, and, of, etc.)
4. **Punctuation removal**: Clean up punctuation marks

In [160]:
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
import string

In [161]:
def preprocess_text_for_bm25(text):
    """
    Preprocesses text for BM25 keyword search.
    
    Steps:
    1. Convert to lowercase for case-insensitive matching
    2. Split into individual tokens (words)  
    3. Remove punctuation from each token
    4. Filter out stop words and empty tokens
    
    Returns:
        List of cleaned tokens suitable for BM25 indexing
    """
    processed_tokens = []
    
    # Split text and process each token
    for token in text.lower().split():
        # Remove punctuation from token
        clean_token = token.strip(string.punctuation)
        
        # Keep token if it's not empty and not a stop word
        if len(clean_token) > 0 and clean_token not in _stop_words.ENGLISH_STOP_WORDS:
            processed_tokens.append(clean_token)
    
    return processed_tokens

# Test the tokenizer
sample_text = "Formula One cars are the world's fastest racing cars!"
tokens = preprocess_text_for_bm25(sample_text)
print(f"Original: {sample_text}")
print(f"Tokens: {tokens}")

Original: Formula One cars are the world's fastest racing cars!
Tokens: ['formula', 'cars', "world's", 'fastest', 'racing', 'cars']


In [162]:
# Tokenize all document chunks for BM25 indexing
print("Preprocessing document corpus for BM25...")

tokenized_document_corpus = []
for chunk in tqdm(document_chunks, desc="Tokenizing"):
    tokens = preprocess_text_for_bm25(chunk)
    tokenized_document_corpus.append(tokens)

print(f"Tokenized {len(tokenized_document_corpus)} document chunks")
print(f"Average tokens per chunk: {np.mean([len(tokens) for tokens in tokenized_document_corpus]):.1f}")

Preprocessing document corpus for BM25...


Tokenizing: 100%|██████████| 35/35 [00:00<00:00, 78293.67it/s]

Tokenized 35 document chunks
Average tokens per chunk: 12.2


In [163]:
# Build BM25 index from tokenized corpus
print("Building BM25 search index...")

bm25_search_index = BM25Okapi(tokenized_document_corpus)

print(f"BM25 index created successfully!")
print(f"Indexed {len(tokenized_document_corpus)} documents")
print(f"Ready for keyword-based search")

Building BM25 search index...
BM25 index created successfully!
Indexed 35 documents
Ready for keyword-based search


In [166]:
def keyword_search_engine(user_query, top_k=3, num_candidates=15):
    """
    BM25-based keyword search engine.
    
    Parameters:
    -----------
    user_query : str
        The search query
    top_k : int
        Number of top results to return
    num_candidates : int
        Number of candidates to consider before ranking
    
    Returns:
    --------
    pd.DataFrame
        Results with columns: ['chunk_text', 'bm25_score', 'chunk_id']
    """
    print(f"Keyword Search Query: {user_query}")
    
    # Tokenize the query using same preprocessing as corpus
    query_tokens = preprocess_text_for_bm25(user_query)
    print(f"Query tokens: {query_tokens}")
    
    # Get BM25 scores for all documents
    bm25_scores = bm25_search_index.get_scores(query_tokens)
    
    # Get top candidates by partitioning (faster than full sort)
    top_candidate_indices = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    
    # Create list of candidates with scores
    candidate_results = [
        {'chunk_id': idx, 'bm25_score': bm25_scores[idx]} 
        for idx in top_candidate_indices
    ]
    
    # Sort candidates by score (highest first)
    ranked_results = sorted(candidate_results, key=lambda x: x['bm25_score'], reverse=True)
    
    # Get top-k results
    top_results = ranked_results[:top_k]
    
    # Create structured results
    results_df = pd.DataFrame([
        {
            'chunk_text': document_chunks[hit['chunk_id']],
            'bm25_score': hit['bm25_score'],
            'chunk_id': hit['chunk_id']
        }
        for hit in top_results
    ])
    
    print(f"Top {len(results_df)} BM25 results:")
    for _, row in results_df.iterrows():
        print(f"     Text: {row['chunk_text'][:100]}...")
        
    return results_df

In [167]:
# Test keyword search with our evaluation queries

for query in evaluation_queries:  
    results = keyword_search_engine(query, top_k=1)
    print("-" * 40)

Keyword Search Query: Who was the most successful F1 driver in the 1950s?
Query tokens: ['successful', 'f1', 'driver', '1950s']
Top 1 BM25 results:
     Text: Stirling Moss, despite often being regarded as one of the greatest Formula One drivers in the 1950s ...
----------------------------------------
Keyword Search Query: When was the Constructors' Championship introduced?
Query tokens: ['constructors', 'championship', 'introduced']
Top 1 BM25 results:
     Text: A Constructors' Championship was added in the 1958 season....
----------------------------------------
Keyword Search Query: Who won the first world championship race?
Query tokens: ['won', 'world', 'championship', 'race']
Top 1 BM25 results:
     Text: The first world championship race, the 1950 British Grand Prix, took place at Silverstone Circuit in...
----------------------------------------
Keyword Search Query: Which manufacturer introduced new engine technology in the 1950s?
Query tokens: ['manufacturer', 'introduced'

## Hybrid Search: Combining Keyword and Semantic Approaches

Hybrid search combines the strengths of both approaches:

**Two-Stage Pipeline:**
1. **Stage 1 (Retrieval)**: BM25 finds initial candidates based on keyword matching
2. **Stage 2 (Reranking)**: Neural reranker reorders results by semantic relevance

**Benefits:**
- **Recall**: BM25 ensures we don't miss exact keyword matches
- **Precision**: Neural reranking improves relevance of final results
- **Efficiency**: Cheaper than pure semantic search at scale
- **Robustness**: Handles both keyword and semantic queries well

In [168]:
def hybrid_search_engine(user_query, top_k=3, num_candidates=10):
    """
    Hybrid search combining BM25 retrieval with neural reranking.
    
    Two-stage process:
    1. BM25 finds initial candidates based on keyword matching
    2. Neural reranker orders results by semantic relevance
    
    Parameters:
    -----------
    user_query : str
        The search query
    top_k : int
        Number of final results to return
    num_candidates : int
        Number of BM25 candidates to rerank
    
    Returns:
    --------
    pd.DataFrame
        Reranked results with columns: ['chunk_text', 'rerank_score', 'bm25_score', 'chunk_id']
    """
    print(f"Hybrid Search Query: {user_query}")
    
    # Stage 1: BM25 Retrieval
    print(f"\nStage 1: BM25 retrieval (finding {num_candidates} candidates)")
    query_tokens = preprocess_text_for_bm25(user_query)
    bm25_scores = bm25_search_index.get_scores(query_tokens)
    
    # Get top candidates
    top_indices = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_candidates = [
        {'chunk_id': idx, 'bm25_score': bm25_scores[idx]} 
        for idx in top_indices
    ]
    bm25_candidates = sorted(bm25_candidates, key=lambda x: x['bm25_score'], reverse=True)
    
    print(f"BM25 top-3 candidates:")
    for i, hit in enumerate(bm25_candidates[:3]):
        chunk_text = document_chunks[hit['chunk_id']]
        print(f"  {i+1}. Score: {hit['bm25_score']:.3f}")
        print(f"     Text: {chunk_text[:80]}...")
    
    # Stage 2: Neural Reranking
    print(f"\nStage 2: Neural reranking ({len(bm25_candidates)} candidates)")
    candidate_texts = [document_chunks[hit['chunk_id']] for hit in bm25_candidates]
    
    rerank_response = cohere_client.rerank(
        query=user_query,
        documents=candidate_texts,
        top_n=top_k,
        return_documents=True
    )
    
    # Combine results
    final_results = []
    for result in rerank_response.results:
        original_candidate = bm25_candidates[result.index]
        final_results.append({
            'chunk_text': result.document.text,
            'rerank_score': result.relevance_score,
            'bm25_score': original_candidate['bm25_score'],
            'chunk_id': original_candidate['chunk_id']
        })
    
    results_df = pd.DataFrame(final_results)
    
    print(f"Final reranked top-{len(results_df)} results:")
    for idx, row in results_df.iterrows():
        print(f"  {idx+1}. Rerank: {row['rerank_score']:.3f}, BM25: {row['bm25_score']:.3f}")
        print(f"     Text: {row['chunk_text'][:80]}...")
    
    return results_df

In [169]:
# Test hybrid search with one challenging query
test_query = "Who was the most successful F1 driver in the 1950s?"
print("Testing hybrid search with challenging query:")
print("=" * 60)

results = hybrid_search_engine(test_query, top_k=3, num_candidates=8)

Testing hybrid search with challenging query:
Hybrid Search Query: Who was the most successful F1 driver in the 1950s?

Stage 1: BM25 retrieval (finding 8 candidates)
BM25 top-3 candidates:
  1. Score: 3.159
     Text: Stirling Moss, despite often being regarded as one of the greatest Formula One d...
  2. Score: 2.750
     Text: Formula One (F1) is the highest class of worldwide racing for open-wheel single-...
  3. Score: 2.165
     Text: Fangio won 24 of the 52 races he entered—still the record for the highest Formul...

Stage 2: Neural reranking (8 candidates)
Final reranked top-3 results:
  1. Rerank: 0.737, BM25: 3.159
     Text: Stirling Moss, despite often being regarded as one of the greatest Formula One d...
  2. Rerank: 0.219, BM25: 2.165
     Text: Fangio won 24 of the 52 races he entered—still the record for the highest Formul...
  3. Rerank: 0.171, BM25: 1.893
     Text: This set the record for the most World Championships won by a single driver, a r...


# RAG

### using cohere

In [174]:
# Lets try the hardest question:
query = "Who was the most successful F1 driver in the 1950s?"
results = semantic_search_engine(query)
docs_dict = [{'text': chunk_text} for chunk_text in results['chunk_text']]
response = co.chat(
    message=query,
    documents=docs_dict
)

🔍 Semantic Search Query: 'Who was the most successful F1 driver in the 1950s?'
📊 Searching through 35 document chunks...
🎯 Query embedding generated: 1024 dimensions
✅ Found 3 relevant results
📏 Similarity scores range: 0.795 - 1.035
💡 Lower scores = higher similarity

🏆 Top Results:
   1. Score: 0.795
      Text: "Stirling Moss, despite often being regarded as one of the greatest Formula One drivers in the 1950s ..."

   2. Score: 1.028
      Text: "Giuseppe Farina, competing for Alfa Romeo, won the first Drivers' World Championship, narrowly defea..."

   3. Score: 1.035
      Text: "Mercedes drivers won the championship for the next two years, before the team withdrew from all moto..."



### using local models

In [121]:
import os
model_path = os.path.expanduser("~/Downloads/Phi-3-mini-4k-instruct-q4.gguf")

from langchain import LlamaCpp
llm = LlamaCpp(
    model_path=model_path,
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

llama_context: n_batch is less than GGML_KQ_MASK_PAD - increasing to 64
llama_context: n_ctx_per_seq (2048) < n_ctx_train (4096) -- the full capacity of the model will not be utilized
ggml_metal_init: skipping kernel_get_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_1row              (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_l4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_bf16                  (not supported)
ggml_metal_init: skipping kernel_mul_mv_id_bf16_f32                (not supported)
ggml_metal_init: skipping kernel_mul_mm_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mm_id_bf16_f32                (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16_h64           (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16_h80           (n

In [175]:
from langchain.embeddings.huggingface import HuggingFaceBgeEmbeddings
embedding_model = HuggingFaceBgeEmbeddings(
    model_name='thenlper/gte-small'
)
from langchain.vectorstores import FAISS
db = FAISS.from_texts(document_chunks, embedding_model)

/opt/miniconda3/envs/re/lib/python3.10/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [176]:
from langchain import PromptTemplate

template = """<|user|>
Relevant information: 
{context}
Provide a concise answer to the following question using the relevant information provided above:
{question}<|end|>
<assistant>"""
prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

In [177]:
from langchain.chains import RetrievalQA

rag = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=db.as_retriever(),
    chain_type_kwargs={
        "prompt": prompt
    },
    verbose=True
)

In [178]:
rag.invoke(query)



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Who was the most successful F1 driver in the 1950s?',
 'result': "\nThe most successful F1 driver in the 1950s was Juan Manuel Fangio, who won five World Drivers' Championships between 1954 and 1957."}

In [179]:
# Bonus question:
query = "Who was more successful: Moss or Fangio?"
rag.invoke(query)



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Who was more successful: Moss or Fangio?',
 'result': '\nBased on the information provided, Juan Manuel Fangio was more successful. He won the Formula One championship five times (1951, 1954, 1955, 1956, and 1957) while Stirling Moss never won the championship but finished second four times between 1955 and 1961. Additionally, Fangio had a higher winning percentage with 24 out of 52 races entered compared to no wins for Moss in the championships.'}